# Module 11 (Option 4): LLM-assisted Forecast Explanations

This notebook demonstrates generating:
- a forecast (baseline)
- a natural language explanation (template or LLM)

If you set `LLM_API_KEY`, the explainer will attempt LLM mode; otherwise it uses a safe template.


In [ ]:
import os
from pathlib import Path
import pandas as pd

import sys
sys.path.append('..')

from src.data.loaders import load_sales_data
from src.models.baseline import moving_average_forecast
from src.explanations import ExplanationContext, explain_forecast

print('Imports OK')


In [ ]:
raw_path = Path('../data/raw/sample_sales.csv')
df = load_sales_data(raw_path)
print(df.shape)

sku_id = df['sku_id'].iloc[0]
df_sku = df[df['sku_id'] == sku_id].sort_values('date')

cutoff = df_sku['date'].max()
horizon = 14

train = df_sku[df_sku['date'] <= cutoff]
y_train = train['units_sold'].astype(float)

forecast = moving_average_forecast(y_train, horizon=horizon, window=7).y_pred.tolist()
forecast_dates = pd.date_range(start=cutoff + pd.Timedelta(days=1), periods=horizon, freq='D')

ctx = ExplanationContext(
    sku_id=sku_id,
    cutoff_date=cutoff,
    horizon=horizon,
    forecast=[float(x) for x in forecast],
    forecast_dates=list(forecast_dates),
    method='moving_average',
    top_drivers=None,
)

result = explain_forecast(ctx, history=train[['date','units_sold'] + ([c for c in ['promotion_flag','price'] if c in train.columns])])
print('Mode:', result.mode)
print('\nExplanation:\n', result.explanation)
print('\nBullets:')
for b in result.bullets:
    print('-', b)
